# 🎤 Speech Enhancement — Step 1: Data Preparation

This notebook:
1. Mounts Google Drive
2. Installs required packages
3. Clones / copies the source code
4. Loads raw voice + noise audio files from your Drive
5. Creates noisy-voice / clean-voice spectrogram pairs and saves them for training

### 📁 Expected Drive layout
```
MyDrive/speech_enhancement/
├── data/
│   ├── voice/   ← clean speech .wav files  (e.g. LibriSpeech)
│   └── noise/   ← background noise .wav files  (e.g. ESC-50)
```

> **Tip:** You only need a handful of files (≥ 10 voice + 10 noise) to do a quick smoke-test with `NB_SAMPLES = 200`.

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile scikit-learn

In [ ]:
# ── 3. Clone repo & add src/ to path ──────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/YOUR_USERNAME/speech_enhancement_alt.git'  # <── update
REPO_DIR = '/content/speech_enhancement_alt'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready, src/ on path ✓')

In [ ]:
# ── 4. (Optional) Override config for quick smoke-test ────────────────────
# Import config first, then monkey-patch any values you want to change.
import config as C

# Uncomment to use fewer samples for a quick test:
# C.NB_SAMPLES = 200

print('Config loaded.')
print(f'  DRIVE_ROOT  : {C.DRIVE_ROOT}')
print(f'  NOISE_DIR   : {C.NOISE_DIR}')
print(f'  VOICE_DIR   : {C.VOICE_DIR}')
print(f'  NB_SAMPLES  : {C.NB_SAMPLES}')

In [ ]:
# ── 5. Verify data directories exist ──────────────────────────────────────
import os

for d in [C.NOISE_DIR, C.VOICE_DIR]:
    files = [f for f in os.listdir(d) if not f.startswith('.')]
    print(f'{d}: {len(files)} files')

In [ ]:
# ── 6. Run data preparation ────────────────────────────────────────────────
from prepare_data import create_data

create_data()   # uses values from config.py

In [ ]:
# ── 7. Quick sanity-check: visualise a spectrogram pair ───────────────────
import numpy as np
import matplotlib.pyplot as plt

noisy = np.load(os.path.join(C.SPEC_DIR, 'noisy_voice_amp_db.npy'))
clean = np.load(os.path.join(C.SPEC_DIR, 'voice_amp_db.npy'))

idx = 0
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(noisy[idx], origin='lower', aspect='auto', cmap='magma')
axes[0].set_title('Noisy Voice')
axes[1].imshow(clean[idx], origin='lower', aspect='auto', cmap='magma')
axes[1].set_title('Clean Voice')
axes[2].imshow(noisy[idx] - clean[idx], origin='lower', aspect='auto', cmap='magma')
axes[2].set_title('Noise Model (target)')
for ax in axes:
    ax.set_xlabel('Time frame')
    ax.set_ylabel('Frequency bin')
plt.suptitle(f'Sample #{idx}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Total samples saved: {noisy.shape[0]}')

In [ ]:
# ── 8. Listen to QC audio ─────────────────────────────────────────────────
import IPython.display as ipd
import soundfile as sf

def play_wav(path):
    data, sr = sf.read(path)
    return ipd.Audio(data, rate=sr)

print('Noisy voice:')
display(play_wav(os.path.join(C.SOUND_DIR, 'noisy_voice_long.wav')))
print('Clean voice:')
display(play_wav(os.path.join(C.SOUND_DIR, 'clean_voice_long.wav')))
print('Noise only:')
display(play_wav(os.path.join(C.SOUND_DIR, 'noise_long.wav')))